# Chronos en zero-shot y PatchTST

Notebook opcional del **módulo 10** del curso (en la app: `uv run dev` y la ruta `#/modulo/transformers-series`). Usa modelos reales; el módulo interactivo explica la teoría con NumPy.

> **Requisitos:** desde la raíz del repositorio, `uv sync --group notebooks` y abre este notebook con `uv run --group notebooks jupyter lab`. La primera ejecución descarga los modelos de Hugging Face.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm
import torch
from chronos import ChronosPipeline

serie = sm.datasets.co2.load_pandas().data["co2"].resample("MS").mean().interpolate().values
contexto, H = serie[:-24], 24

In [ ]:
pipe = ChronosPipeline.from_pretrained("amazon/chronos-t5-small", device_map="cpu", torch_dtype=torch.float32)
muestras = pipe.predict(torch.tensor(contexto), prediction_length=H, num_samples=50)[0].numpy()
q10, q50, q90 = np.quantile(muestras, [0.1, 0.5, 0.9], axis=0)
mae = np.abs(q50 - serie[-H:]).mean()
escala = np.abs(np.diff(contexto)).mean()
print("MASE de Chronos (zero-shot):", round(mae / escala, 3))

In [ ]:
t = np.arange(len(serie))
plt.plot(t[-120:], serie[-120:], color="k")
plt.fill_between(t[-H:], q10, q90, alpha=0.3)
plt.plot(t[-H:], q50)
plt.show()

PatchTST en `transformers`: un modelo pequeño configurado a mano (para entrenarlo, usa un `Trainer` con ventanas como en el módulo 9).

In [ ]:
from transformers import PatchTSTConfig, PatchTSTForPrediction

config = PatchTSTConfig(num_input_channels=1, context_length=96, patch_length=16, patch_stride=8,
                        prediction_length=24, d_model=64, num_attention_heads=4, num_hidden_layers=2)
modelo = PatchTSTForPrediction(config)
x = torch.tensor(contexto[-96:], dtype=torch.float32)[None, :, None]
print(modelo(past_values=x).prediction_outputs.shape)